---

## 모델 호출 재시도 미들웨어

``@wrap_model_call`` 로 ``handler(request)`` 를 감싸면, 모델 API 오류 시 미들웨어에서 재시도 로직을 넣을 수 있습니다.

**미들웨어 활용 사례:**
- 일시적 네트워크·레이트 리밋 오류 복구
- 모델 호출 실패 시 지수 백오프(이 데모는 고정 횟수 재시도)
- 호출 전후 로깅·메트릭 수집

기본 ``MiddlewareModelRetryAgent()`` 는 **존재하지 않는 모델명** ``gpt-4.1-mini`` 를 써서 재시도 메시지를 확인합니다.

**미들웨어 훅 요약:**

| 훅 | 시점 | 이 노트북에서 하는 일 |
|:---|:---|:---|
| ``wrap_model_call`` | 각 LLM 호출을 감쌀 때 | ``handler`` 실패 시 최대 ``max_retries`` 회 재시도 |
| ``make_model_retry_middleware`` | 팩토리 | ``max_retries`` (기본 3) 주입 |
| ``MiddlewareModelRetryAgent`` | 에이전트 | 기본 잘못된 모델명 또는 유효 모델로 ``create_agent`` |

``make_model_retry_middleware(max_retries=...)`` 로 재시도 횟수를 바꿀 수 있습니다.


In [1]:
from feature.MiddlewareModelRetry import MiddlewareModelRetryAgent
from langchain_core.messages import HumanMessage

# 일부러 존재하지 않는 모델명 → 재시도 로그 확인용
workflow_retry_demo = MiddlewareModelRetryAgent()

아래 셀은 모델명 오류로 ``handler`` 가 실패하고, ``오류 발생으로 1/3 …`` 로그가 출력된 뒤 **3회 모두 실패하면 예외**가 납니다.

In [2]:
# invoke: wrap_model_call 재시도 로그 → (최종 실패 시 예외)
try:
    _ = workflow_retry_demo.invoke(
        inputs={"messages": [HumanMessage(content="대한민국 수도")]},
    )
except Exception as e:
    print(f"\n최종 실패 (예상): {type(e).__name__}: {e}")

오류 발생으로 1/3 번째 재시도합니다: test
오류 발생으로 2/3 번째 재시도합니다: test

최종 실패 (예상): Exception: test
